In [1]:
import sys
#!{sys.executable} -m pip install --upgrade --force-reinstall "git+https://github.com/hms-dbmi/pic-sure-python-adapter-hpds.git@main"
!{sys.executable} -m pip install -e /Users/george/code_workspaces/bdc/pic-sure-python-adapter-hpds

Obtaining file:///Users/george/code_workspaces/bdc/pic-sure-python-adapter-hpds
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for picsure (pyproject.toml) ... done
  Created wheel for picsure: filename=picsure-0.1.0-py3-none-any.whl size=6079 sha256=0d75f228aa77913a466ce0c9dbac6bca90f9d18024b9b88b84a27213a3ed98cd
  Stored in directory: /private/var/folders/n3/wd1bprj14gjf3l0y580nx2z40000gq/T/pip-ephem-wheel-cache-rrtkcnvi/wheels/e6/09/c8/9b45f1c9855c43c8e64249ed1b6967e0f032bc8f5b5968a868
Successfully built picsure
  Attempting uninstall: picsure
    Found existing installation: picsure 0.1.0
    Uninstalling picsure-0.1.0:
      Successfully uninstalled picsure-0.1.0

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --

In [2]:
import picsure

In [3]:
token_file = "token.txt"

with open(token_file, "r") as f:
    my_token = f.read()

In [27]:
authorized_session = picsure.connect(
    picsure.Platform.BDC_DEV_AUTHORIZED,
    my_token,
    dev_mode=True
)

picsure.http /psama/user/me 200 308ms in=0B out=24.6KB retry=0
picsure.http /picsure/info/resources 200 132ms in=0B out=264B retry=0
picsure.http /psama/user/me/queryTemplate/ 200 114ms in=0B out=11.0KB retry=0
picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=1 200 129ms in=6.3KB out=635B retry=0
picsure.connect connect 0ms


You're successfully connected to BDC Authorized as user george_colon@hms.harvard.edu!
Your token expires on unknown.


In [28]:
all_facets = authorized_session.showAllFacets()
all_facets

picsure.http /picsure/proxy/dictionary-api/facets 200 323ms in=6.3KB out=110.8KB retry=0
picsure.fn session.showAllFacets 327ms


,category,Category Display,display,description,value,count
0,dataset_id,Dataset,RECOVER_Adult (phs003463),Researching COVID to Enhance Recovery (RECOVER...,phs003463,160676
1,dataset_id,Dataset,FHS (phs000007),Framingham Cohort,phs000007,54984
2,dataset_id,Dataset,BioLINCC_FHS (phs003594),Framingham Heart Study (FHS) BioLINCC,phs003594,34808
3,dataset_id,Dataset,ARIC (phs000280),Atherosclerosis Risk in Communities (ARIC) Cohort,phs000280,26106
4,dataset_id,Dataset,RECOVER_Pediatric (phs003461),Researching COVID to Enhance Recovery (RECOVER...,phs003461,20680
...,...,...,...,...,...,...
480,data_source,Data Type,Genomic,Associated with genomic data,data_source_genomic,259
481,data_source,Data Type,Biosamples,Associated with biosample data,data_source_biosamples,145
482,data_source,Data Type,Electrocardiogram,Associated with electrocardiogram data,data_source_electrocardiogram,0
483,data_type,Type of Variable,Categorical,,categorical,282035


In [29]:
#all_facets[all_facets['value'].str.contains('biolincc')]
#all_facets[all_facets['value'] == 'tutorial-biolincc_framingham']
tutorial_biolincc_framingham_facet = authorized_session.facets()
tutorial_biolincc_framingham_facet.add('dataset_id', 'tutorial-biolincc_framingham')

picsure.http /picsure/proxy/dictionary-api/facets 200 294ms in=6.3KB out=110.8KB retry=0
picsure.fn session.facets 295ms


In [32]:
tutorial_search_results = authorized_session.search("Current cigarette smoking at exam", facets=tutorial_biolincc_framingham_facet)
#tutorial_search_results = authorized_session.search("Current cigarette smoking at exam")
#tutorial_search_results = authorized_session.search("smoke", facets=tutorial_biolincc_framingham_facet)
tutorial_search_results

picsure.http /picsure/proxy/dictionary-api/concepts?page_number=0&page_size=487375 200 263ms in=6.7KB out=697B retry=0
picsure.fn session.search 264ms


,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\tutorial-biolincc_framingham\CURSMOKE\,CURSMOKE,CURSMOKE,Current cigarette smoking at exam,Categorical,tutorial-biolincc_framingham,"[Current smoker, Not current smoker]",None,None,True,None,biolincc_framingham


In [33]:
tutorial_search_results['values']

0    [Current smoker, Not current smoker]
Name: values, dtype: object

In [34]:
cursmoke_var = tutorial_search_results[tutorial_search_results['name'] == 'CURSMOKE']
cursmoke_var

,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\tutorial-biolincc_framingham\CURSMOKE\,CURSMOKE,CURSMOKE,Current cigarette smoking at exam,Categorical,tutorial-biolincc_framingham,"[Current smoker, Not current smoker]",None,None,True,None,biolincc_framingham


In [35]:
raw_values = cursmoke_var['values'].iloc[0]
raw_values

['Current smoker', 'Not current smoker']

In [36]:
cursmoke = picsure.createClause(
    cursmoke_var['conceptPath'],
    type=picsure.ClauseType.FILTER,
    categories=cursmoke_var['values'].iloc[0]
)

In [37]:
query = authorized_session.runQuery(cursmoke, type=picsure.QueryType.COUNT)

picsure.http /picsure/v3/query/sync 200 8257ms in=326B out=4B retry=0
picsure.fn session.runQuery 8258ms


In [39]:
query.value

4434

In [40]:
authorized_session.dev_events()

,timestamp,kind,name,duration_ms,bytes_in,bytes_out,status,retry,error,metadata
0,2026-04-29 17:19:27.446348+00:00,http,/psama/user/me,307.859458,0.0,25233.0,200.0,0,None,{}
1,2026-04-29 17:19:27.579436+00:00,http,/picsure/info/resources,132.227791,0.0,264.0,200.0,0,None,{}
2,2026-04-29 17:19:27.693845+00:00,http,/psama/user/me/queryTemplate/,114.005500,0.0,11263.0,200.0,0,None,{}
3,2026-04-29 17:19:27.823376+00:00,http,/picsure/proxy/dictionary-api/concepts?page_nu...,128.779250,6495.0,635.0,200.0,0,None,{}
4,2026-04-29 17:19:27.823787+00:00,connect,connect,0.000000,NaN,NaN,NaN,0,None,"{'resources': 5, 'consents': 427, 'total_conce..."
5,2026-04-29 17:19:56.335933+00:00,http,/picsure/proxy/dictionary-api/facets,322.759500,6495.0,113474.0,200.0,0,None,{}
6,2026-04-29 17:19:56.340553+00:00,function,session.showAllFacets,327.447750,NaN,NaN,NaN,0,None,"{'df_rows': 485, 'df_cols': 6}"
7,2026-04-29 17:20:14.628833+00:00,http,/picsure/proxy/dictionary-api/facets,293.564042,6495.0,113474.0,200.0,0,None,{}
8,2026-04-29 17:20:14.630551+00:00,function,session.facets,295.383459,NaN,NaN,NaN,0,None,{}
9,2026-04-29 17:20:29.991375+00:00,http,/picsure/proxy/dictionary-api/concepts?page_nu...,367.004666,6877.0,697.0,200.0,0,None,{}
